# Malware hash threat hunt

Join **PE import features** with **MalwareBazaar threat intelligence** on **MD5 hash**, then chart families and suspicious Windows API import flags.

**Data plane:** CSV retrieval and the hunt **join** run in **Cribl Search** (`externaldata` + published `$vt_lookups`, KQL) — not via Pyodide `fetch` and **not** through `config/proxies.yml`.

**Run All** works out of the box: no Auth-Key; TI and PE load from [notebook-app-example-data](https://github.com/michaelhyatt/notebook-app-example-data) via Search. Open this copy from **Welcome → Examples** (not an old saved tab).

## What you will do

| Step | Question it answers |
|------|---------------------|
| Load TI | What hashes did MalwareBazaar see recently? |
| Load PE lookup | Which import flags exist per hash in the teaching PE CSV? |
| Align + publish | What labels (`signature`, type, first seen) attach to each MD5? |
| Hunt join | Which PE profiles share an MD5 with TI? |
| Charts | Which families dominate hits; which imports look hot? |

```
Hosted TI CSV  ──externaldata──►  mb_ti
Hosted PE CSV  ──externaldata──►  PE lookup
                                       └──► TI lookup (labels)
                                                      │
                                                      ▼
                                            KQL inner join on MD5
```

**TI source (default):** hosted [`malwarebazaar_ti_lookup.csv`](https://github.com/michaelhyatt/notebook-app-example-data/blob/main/malware-hunt/malwarebazaar_ti_lookup.csv) (teaching sample, no Auth-Key). **PE source (default):** [`pe_imports_hunt.csv`](https://github.com/michaelhyatt/notebook-app-example-data/blob/main/malware-hunt/pe_imports_hunt.csv) with aligned MD5s. **Optional:** `MALWAREBAZAAR_AUTH_KEY` (live export), `MALWARE_HUNT_TI_CSV_URL` / `MALWARE_HUNT_PE_CSV_URL`, or **Annex** for IEEE/Kaggle PE.

**Related:** `Threat_Hunting_Playbook.ipynb`, `Cribl_Search_Lookup_Magics.ipynb`, `Cribl_API_Examples.ipynb`.

## Prerequisites

1. Hosted Cribl app with **Cribl Search**, `%%cribl_search`, and lookup magics.
2. **No MalwareBazaar Auth-Key** on the default path — TI/PE load from [notebook-app-example-data](https://github.com/michaelhyatt/notebook-app-example-data) via Search.
3. **Optional:** `MALWAREBAZAAR_AUTH_KEY`, `MALWARE_HUNT_TI_CSV_URL`, or `MALWARE_HUNT_PE_CSV_URL` to override hosted files.
4. **No `proxies.yml` changes** — Search workers HTTP GET the raw GitHub URLs (not the app pack `/data/` path).
5. Lookups ≤ **10,000** rows. Heavy Search cells use `timeout=300` (seconds).
6. Run the **Setup** code cell below before §B (defines `load_mb_ti_from_search` and helpers).

## Setup — URLs and CSV helpers

Run this cell first (**Run All** runs it automatically). It defines `TI_CSV_URL`, `PE_CSV_URL`, and `load_mb_ti_from_search` / `load_pe_imports_from_search`.


In [ ]:
# Example datasets: https://github.com/michaelhyatt/notebook-app-example-data
import os

# Hosted CSV URLs — keep in sync with src/domain/exampleDataUrls.ts
EXAMPLE_DATA_BASE = "https://raw.githubusercontent.com/michaelhyatt/notebook-app-example-data/main"
HOSTED_TI_URL = "https://raw.githubusercontent.com/michaelhyatt/notebook-app-example-data/main/malware-hunt/malwarebazaar_ti_lookup.csv"
HOSTED_PE_URL = "https://raw.githubusercontent.com/michaelhyatt/notebook-app-example-data/main/malware-hunt/pe_imports_hunt.csv"

TI_CSV_URL = (os.environ.get("MALWARE_HUNT_TI_CSV_URL") or "").strip() or HOSTED_TI_URL
_mb_auth = (os.environ.get("MALWAREBAZAAR_AUTH_KEY") or "").strip()
if _mb_auth:
    TI_CSV_URL = f"https://mb-api.abuse.ch/v2/files/exports/{_mb_auth}/recent.csv"

_pe_override = (os.environ.get("MALWARE_HUNT_PE_CSV_URL") or "").strip()
PE_CSV_URL = _pe_override or HOSTED_PE_URL
USE_IEEE_PE_HTTP = bool(_pe_override) and _pe_override != HOSTED_PE_URL

print("EXAMPLE_DATA_BASE:", EXAMPLE_DATA_BASE)
print("TI (Search HTTP GET):", TI_CSV_URL)
print("PE (Search HTTP GET):", PE_CSV_URL)
print("USE_IEEE_PE_HTTP (Annex):", USE_IEEE_PE_HTTP)

import io

import pandas as pd

HUNT_IMPORT_COLUMNS = [
    "GetProcAddress",
    "LoadLibraryA",
    "VirtualAlloc",
    "VirtualProtect",
    "WriteProcessMemory",
    "CreateRemoteThread",
    "WinExec",
    "ShellExecuteA",
    "URLDownloadToFileA",
    "InternetOpenA",
    "RegSetValueExA",
    "CreateProcessA",
    "ExitProcess",
    "IsDebuggerPresent",
    "NtQueryInformationProcess",
]


def _pick_col(frame: pd.DataFrame, *names: str) -> str | None:
    keys = {str(c).strip().lower(): c for c in frame.columns}
    for n in names:
        if n in keys:
            return keys[n]
    return None


MB_CSV_HEADER = (
    '"first_seen_utc","sha256_hash","md5_hash","sha1_hash","reporter","file_name",'
    '"file_type_guess","mime_type","signature","clamav","vtpercent","imphash","ssdeep","tlsh"'
)


def _mb_csv_lines_from_search(frame: pd.DataFrame) -> list[str]:
    """Collect CSV lines from Search HTTP / externaldata (_raw / Event)."""
    raw_col = _pick_col(frame, "_raw", "event")
    if raw_col is None:
        return []
    lines: list[str] = []
    for blob in frame[raw_col].astype(str):
        lines.extend(blob.splitlines())
    if len(lines) < 15 and len(frame) == 1 and raw_col is not None:
        blob = str(frame[raw_col].iloc[0])
        if len(blob) > 400:
            lines = blob.splitlines()
    return lines


def _looks_like_md5_series(series: pd.Series) -> bool:
    sample = series.dropna().astype(str).str.strip().str.lower().head(24)
    if sample.empty:
        return False
    return float(sample.str.fullmatch(r"[0-9a-f]{32}").mean()) > 0.5


def _parse_mb_csv_lines(lines: list[str]) -> pd.DataFrame:
    header_line = None
    data_lines: list[str] = []
    for ln in lines:
        s = ln.strip()
        if not s or s.startswith("#"):
            continue
        lower = s.lower()
        if header_line is None and "md5" in lower and "," in s:
            header_line = s.lstrip("#").strip()
            continue
        data_lines.append(s)
    if not header_line:
        header_line = MB_CSV_HEADER
    if not data_lines:
        raise ValueError(
            "MalwareBazaar CSV missing data rows. "
            f"Got {len(lines)} line(s) from Search; re-run §B."
        )
    return pd.read_csv(io.StringIO(header_line + "\n" + "\n".join(data_lines)))


def _normalize_mb_ti_frame(frame: pd.DataFrame) -> pd.DataFrame:
    """MalwareBazaar table from typed CSV columns or line-per-event / full-body _raw."""
    md5_col = _pick_col(frame, "md5", "md5_hash", "hash")
    if md5_col and _looks_like_md5_series(frame[md5_col]):
        out = frame.copy()
    else:
        lines = _mb_csv_lines_from_search(frame)
        if not lines:
            raise ValueError(
                "Expected MalwareBazaar columns or CSV lines in _raw/Event. "
                f"Got columns: {list(frame.columns)}"
            )
        out = _parse_mb_csv_lines(lines)
    out.columns = [str(c).strip().strip('"') for c in out.columns]
    for col in out.columns:
        if out[col].dtype == object:
            out[col] = out[col].astype(str).str.strip().str.strip('"')
    return out


def load_mb_ti_from_search(frame: pd.DataFrame) -> pd.DataFrame:
    out = _normalize_mb_ti_frame(frame)
    if len(out) < 20:
        raise ValueError(
            f"TI load returned only {len(out)} row(s) from Search. "
            f"Re-run §B (URL: {TI_CSV_URL!r})."
        )
    return out


def load_pe_imports_from_search(frame: pd.DataFrame) -> pd.DataFrame:
    """Teaching PE import flags (hash + binary columns) from Search externaldata or HTTP dataset."""
    hash_col = _pick_col(frame, "hash", "md5", "md5_hash")
    if hash_col and _looks_like_md5_series(frame[hash_col]):
        out = frame.copy()
    else:
        lines = _mb_csv_lines_from_search(frame)
        if not lines:
            raise ValueError(
                "Expected PE hash/import columns or CSV lines in _raw/Event. "
                f"Got columns: {list(frame.columns)}"
            )
        out = pd.read_csv(io.StringIO("\n".join(lines)))
    out.columns = [str(c).strip().strip('"') for c in out.columns]
    hash_col = _pick_col(out, "hash", "md5", "md5_hash")
    if hash_col is None:
        raise ValueError(f"PE CSV missing hash column; got {list(out.columns)}")
    if hash_col != "hash":
        out = out.rename(columns={hash_col: "hash"})
    out["hash"] = out["hash"].astype(str).str.strip().str.lower()
    out = out[out["hash"].str.len() == 32]
    for col in HUNT_IMPORT_COLUMNS:
        if col not in out.columns:
            out[col] = 0
    keep = ["hash"] + [c for c in HUNT_IMPORT_COLUMNS if c in out.columns]
    out = out[keep].drop_duplicates(subset=["hash"]).head(2000)
    if len(out) < 20:
        raise ValueError(
            f"PE load returned only {len(out)} row(s). Re-run §C (URL: {PE_CSV_URL!r})."
        )
    return out

def build_hunt_hits_df(pe_df: pd.DataFrame, ti_df: pd.DataFrame) -> pd.DataFrame:
    """Inner join PE hash to TI md5 in pandas (when KQL lookup join returns no rows)."""
    left = pe_df.copy()
    right = ti_df.copy()
    left["hash"] = left["hash"].astype(str).str.strip().str.lower()
    right["md5"] = right["md5"].astype(str).str.strip().str.lower()
    pe_cols = ["hash"] + [c for c in HUNT_IMPORT_COLUMNS if c in left.columns]
    ti_cols = ["md5"] + [c for c in ("malware_family", "signature", "tags") if c in right.columns]
    return left[pe_cols].merge(right[ti_cols], left_on="hash", right_on="md5", how="inner")


def align_hunt_lookups(pe_df: pd.DataFrame, ti_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Keep only MD5s present in both PE and TI lookups (teaching overlap)."""
    pe_df = pe_df.copy()
    ti_df = ti_df.copy()
    pe_df["hash"] = pe_df["hash"].astype(str).str.strip().str.lower()
    ti_df["md5"] = ti_df["md5"].astype(str).str.strip().str.lower()
    overlap = set(pe_df["hash"]) & set(ti_df["md5"])
    if len(overlap) < 10:
        raise ValueError(
            f"Only {len(overlap)} overlapping MD5(s) between PE ({len(pe_df):,} rows) and "
            f"TI ({len(ti_df):,} rows). Re-run §B–§C (hosted example-data URLs)."
        )
    pe_out = pe_df[pe_df["hash"].isin(overlap)].drop_duplicates(subset=["hash"])
    ti_out = ti_df[ti_df["md5"].isin(overlap)].drop_duplicates(subset=["md5"])
    print(f"Aligned lookups: {len(pe_out):,} PE hashes ∩ {len(ti_out):,} TI md5 ({len(overlap):,} overlap)")
    return pe_out, ti_out

print("Helpers loaded (TI + PE CSV normalizers).")


## A0) Reset lookups (idempotent)

Clears prior hunt artifacts so **Run All** is repeatable.


In [ ]:
%%cribl_delete_search_lookup notebook_pe_features.csv

In [ ]:
%%cribl_delete_search_lookup notebook_mb_ti.csv

In [ ]:
%%cribl_api DELETE /m/default_search/search/datasets/notebook_ti_imports var=_ti_ds_a0 ignoreFailure=true response=json

In [ ]:
%%cribl_api DELETE /m/default_search/search/dataset-providers/notebook_ti_http var=_ti_prov_a0 ignoreFailure=true response=json

## B) Load TI CSV (`externaldata`)

Fetches the hosted TI table through Search [`externaldata`](https://docs.cribl.io/search/externaldata/) (same pattern as §C). Avoids MalwareBazaar `recent` CSV comment lines that break some tenants.


In [ ]:
%%cribl_search var=mb_ti_raw lang=kql limit=5000 preview=true earliest=-1h latest=now timeout=300 template=on
externaldata
[
  "{{ TI_CSV_URL }}"
]
with(
  datatype="CSV Datatypes"
)

In [ ]:
mb_ti = load_mb_ti_from_search(mb_ti_raw).head(5000)

md5_col = _pick_col(mb_ti, "md5", "md5_hash", "hash")
sig_col = _pick_col(mb_ti, "signature")
ftype_col = _pick_col(mb_ti, "file_type", "file_type_guess")
exe_n = 0
if ftype_col:
    exe_n = int(mb_ti[ftype_col].astype(str).str.lower().isin(["exe", "dll"]).sum())

print(f"TI rows: {len(mb_ti):,}  |  columns: {list(mb_ti.columns)[:6]}…")
print(f"MD5 column: {md5_col!r}  |  signatures: {sig_col!r}  |  exe/dll rows: {exe_n:,}")
mb_ti.head(5)

## C) Load PE import lookup (hosted CSV)

Loads [`pe_imports_hunt.csv`](https://github.com/michaelhyatt/notebook-app-example-data/blob/main/malware-hunt/pe_imports_hunt.csv) via Search **`externaldata`** (same pattern as the Anomaly notebook). MD5s align with the TI lookup for a reliable inner join.


In [ ]:
%%cribl_search var=pe_raw lang=kql limit=2500 preview=true earliest=-1h latest=now timeout=300 template=on
externaldata
[
  "{{ PE_CSV_URL }}"
]
with(
  datatype="CSV Datatypes"
)

In [ ]:
pe_lookup_df = load_pe_imports_from_search(pe_raw)
print(f"PE rows (pre-align): {len(pe_lookup_df):,}  |  columns: {list(pe_lookup_df.columns)[:5]}…")
pe_lookup_df.head(8)


## D) Align MD5s and publish lookups

Inner join in §F needs the same MD5 on both sides. We intersect PE `hash` and TI `md5` in Python, then **`%%cribl_save_search_lookup`** both tables.


In [ ]:
md5_col = _pick_col(mb_ti, "md5", "md5_hash", "hash")
if md5_col is None:
    raise ValueError(f"Expected md5 column; got {list(mb_ti.columns)}")

mb_lookup_df = pd.DataFrame()
mb_lookup_df["md5"] = mb_ti[md5_col].astype(str).str.strip().str.lower()
for col in (
    "signature",
    "tags",
    "malware_family",
    "file_type",
    "file_type_guess",
    "first_seen",
    "first_seen_utc",
):
    src_col = _pick_col(mb_ti, col)
    if src_col:
        mb_lookup_df[col] = mb_ti[src_col]

mb_lookup_df = mb_lookup_df.dropna(subset=["md5"])
mb_lookup_df = mb_lookup_df[mb_lookup_df["md5"].str.len() == 32]
mb_lookup_df = mb_lookup_df.drop_duplicates(subset=["md5"]).head(10000)

if "malware_family" not in mb_lookup_df.columns and "signature" in mb_lookup_df.columns:
    mb_lookup_df["malware_family"] = mb_lookup_df["signature"].astype(str)

pe_lookup_df, mb_lookup_df = align_hunt_lookups(pe_lookup_df, mb_lookup_df)
print("TI lookup rows:", len(mb_lookup_df))
mb_lookup_df.head(8)


In [ ]:
%%cribl_save_search_lookup notebook_pe_features.csv var=pe_lookup_df replace=true


In [ ]:
%%cribl_save_search_lookup notebook_mb_ti.csv var=mb_lookup_df replace=true


In [ ]:
%%cribl_search var=lookup_preview lang=kql limit=10 preview=true earliest=-1h latest=now
dataset="$vt_lookups" lookupFile="notebook_mb_ti"
| limit 10

## F) Hunt — join PE lookup to TI lookup on MD5

Do not start with `let` — `%%cribl_search` prepends `cribl` only when the body begins with `dataset=`.

In [ ]:
%%cribl_search var=hunt_hits_kql lang=kql limit=5000 preview=true earliest=-1h latest=now timeout=300
dataset="$vt_lookups" lookupFile="notebook_pe_features"
| extend hash = tolower(tostring(hash))
| join kind=inner (
    dataset="$vt_lookups" lookupFile="notebook_mb_ti"
    | extend md5 = tolower(tostring(md5))
  ) on $left.hash == $right.md5
| project hash, malware_family, signature, tags, GetProcAddress, VirtualAlloc, WriteProcessMemory, CreateRemoteThread, URLDownloadToFileA
| limit 2000

In [ ]:
hunt_hits_pd = build_hunt_hits_df(pe_lookup_df, mb_lookup_df)
kql_n = len(hunt_hits_kql) if "hunt_hits_kql" in globals() else 0
if kql_n > 0:
    hunt_hits = hunt_hits_kql.copy()
else:
    hunt_hits = hunt_hits_pd.copy()
print(f"Hunt rows: {len(hunt_hits):,} (KQL={kql_n:,}, pandas={len(hunt_hits_pd):,})")
if len(hunt_hits) == 0:
    raise ValueError(
        "No join hits after KQL and pandas fallback. Re-run A0 then Run All; check §D overlap."
    )

## G) Checkpoint — match rate

In [ ]:
%%cribl_search var=pe_total lang=kql limit=0 preview=false earliest=-1h latest=now timeout=300
dataset="$vt_lookups" lookupFile="notebook_pe_features"
| summarize pe_samples = dcount(hash)

In [ ]:
hits = len(hunt_hits)
pe_n = int(pe_total.iloc[0]["pe_samples"]) if len(pe_total) else len(pe_lookup_df)
rate = (hits / pe_n * 100) if pe_n else 0
print(f"Join hits: {hits:,} / distinct PE hashes: {pe_n:,} ({rate:.1f}%)")
if hits == 0:
    raise ValueError("No join hits — re-run A0 then Run All.")
hunt_hits.head(10)

## H) Visualize TI families and suspicious imports

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

SUSPICIOUS = [
    "VirtualAlloc",
    "WriteProcessMemory",
    "CreateRemoteThread",
    "URLDownloadToFileA",
    "WinExec",
    "IsDebuggerPresent",
]

fam_col = next(
    (c for c in hunt_hits.columns if str(c).lower() in ("malware_family", "signature")),
    None,
)
if fam_col is None:
    raise ValueError(f"Expected malware_family or signature in hunt_hits; got {list(hunt_hits.columns)}")

by_family = hunt_hits[fam_col].value_counts().head(8)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
by_family.sort_values().plot(kind="barh", ax=axes[0], color="#c44e52")
axes[0].set_title("Hunt hits by malware family")
axes[0].set_xlabel("Samples")

present = [c for c in SUSPICIOUS if c in hunt_hits.columns]
if present:
    means = hunt_hits[present].apply(pd.to_numeric, errors="coerce").fillna(0).mean().sort_values(ascending=False)
    means.plot(kind="bar", ax=axes[1], color="#4c72b0")
    axes[1].set_title("Mean import flag rate (matched samples)")
    axes[1].set_ylabel("Fraction with flag = 1")
    axes[1].tick_params(axis="x", rotation=45)
else:
    axes[1].text(0.5, 0.5, "Import columns not in projection", ha="center")

plt.tight_layout()
plt.show()
print("Interpretation: inner join surfaces PE rows with both import features and TI labels on the same MD5.")

## Interpretation

- **Inner join** answers: *which PE import profiles in our sample also have MalwareBazaar-style TI on the same hash?*
- **Match rate** on the teaching path is typically **≥60%** (curated overlap in the example-data repo). With real IEEE PE imports + live MalwareBazaar, overlap is usually lower.
- **MD5 reuse** — one hash can map to multiple submissions over time; treat TI as contextual, not ground truth.
- **Production** — use MalwareBazaar public or authenticated exports for TI; load real PE features from IEEE/Kaggle or your feature store (Annex / `MALWARE_HUNT_PE_CSV_URL`).

## Cleanup

In [ ]:
%%cribl_delete_search_lookup notebook_mb_ti.csv

In [ ]:
%%cribl_delete_search_lookup notebook_pe_features.csv

In [ ]:
%%cribl_api DELETE /m/default_search/search/datasets/notebook_pe_imports var=_ds_del ignoreFailure=true response=json

In [ ]:
%%cribl_api DELETE /m/default_search/search/dataset-providers/notebook_pe_http var=_prov_del ignoreFailure=true response=json

In [ ]:
%%cribl_api DELETE /m/default_search/search/datasets/notebook_ti_imports var=_ti_ds_del ignoreFailure=true response=json

In [ ]:
%%cribl_api DELETE /m/default_search/search/dataset-providers/notebook_ti_http var=_ti_prov_del ignoreFailure=true response=json

## Annex — IEEE/Kaggle PE imports via HTTP provider

When `USE_IEEE_PE_HTTP` is true (`MALWARE_HUNT_PE_CSV_URL` or uncommented `PE_CSV_URL` in the URL cell), run §A0 then the cells below, replace §C with a preview of `notebook_pe_imports`, and in §F join `dataset="notebook_pe_imports"` to `notebook_mb_ti` instead of `notebook_pe_features`.


In [ ]:
if not USE_IEEE_PE_HTTP:
    print("Annex skipped — default path uses hosted pe_imports_hunt.csv in §C.")
else:
    print("Annex: registering IEEE/custom PE HTTP dataset (run cells below).")


### Annex — manual PE HTTP provider (custom `MALWARE_HUNT_PE_CSV_URL` only)

When `USE_IEEE_PE_HTTP` is true after the URL cell, run these **`%%cribl_api`** cells yourself (they are not part of default **Run All**):

```
%%cribl_api POST /m/default_search/search/dataset-providers var=pe_provider_res response=json template=on
json:
  type: api_http
  id: notebook_pe_http
  description: PE import features (malware hash hunt example)
  authenticationMethod: none
  availableEndpoints:
    - name: pe_imports
      method: GET
      url: "{{ PE_CSV_URL }}"
      headers: []
      dataField: ""```


```
%%cribl_api POST /m/default_search/search/datasets var=pe_dataset_res response=json
json:
  type: api_http
  id: notebook_pe_imports
  description: PE imports sample for hash threat hunt
  provider: notebook_pe_http
  enabledEndpoints:
    - pe_imports
  filter: "true"
  searchVersion: v1
  breakerRulesets:
    - Cribl Search
  metadata:
    enableAcceleration: false
```

Then query `dataset="notebook_pe_imports"` and join to `notebook_mb_ti` in §F instead of `notebook_pe_features`.


## Troubleshooting

| Symptom | Fix |
|---------|-----|
| Only `#` comment rows / no data | Do not point `MALWARE_HUNT_TI_CSV_URL` at live MalwareBazaar `recent` (comment preamble breaks Search). Use default hosted URL. |
| `Expected md5 column` / bad columns | Re-run **§B** + parse cell — normalizer rebuilds from `_raw` when typed columns are wrong. |
| Provider/dataset 409 | Run **A0** deletes, then **§B** again. |
| No join hits | §D must run **`%%cribl_save_search_lookup`** for both `notebook_pe_features` and `notebook_mb_ti`; check align printed ≥10 overlap. |
| `load_mb_ti_from_search` / `load_pe_imports_from_search` missing | Re-open example from **Welcome** (stale tab). |
| Stale notebook tab | Close tab → **Welcome** → **Malware Hash Threat Hunt** → **Open example**. |